## Prepare the Settings

In [ ]:
import os
import errno
import logging
from datetime import datetime
from omegaconf import OmegaConf

import RETFound_MAE.models_vit as models_vit
from dataclasses import dataclass
from typing import Optional, Sequence, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, SequentialSampler, RandomSampler, Subset
from torch.optim import AdamW, Optimizer
from torch.optim.lr_scheduler import LambdaLR

from dataset import Fundus_RiskFactor_Dataset
from model import Model_Wrapper
from utils import mkdir, get_cosine_schedule_with_warmup
from losses import compute_similarity_label, sigmoid_loss, weighted_bce_loss, Thresholds, align_sims_k, build_M, estimate_beta_grid

In [ ]:
def set_seed(seed=42):
    import os
    import random
    import numpy as np
    import torch

    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --- PATCH: utilities for Sit/Sii/Stt, M, and β ---

@dataclass
class Thresholds:
    p2: float   # Sii >
    p3: float   # Stt >

@torch.no_grad()
def _cosine_sim(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # assumes a,b row-normalized
    return a @ b.t()

@dataclass
class ThresholdsST:
    p2: float   # threshold for Sii
    p3: float   # threshold for Stt

@torch.no_grad()
def _align_sims_k(
    Xf: torch.Tensor,                # (N_img, d), L2-normalized
    Tf: torch.Tensor,                # (N_txt, d), L2-normalized, N_txt = k * N_img
    k: Optional[int] = None,
    caption_groups: Optional[Sequence[Sequence[int]]] = None,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build ONLY Sii and Stt aligned to (N_img, N_txt).
    - Sii: replicate image-image sims across each caption group of image j
    - Stt: average k×k text-text sims between image i's captions and image j's captions
    Returns:
        Sii, Stt with shape (N_img, N_txt)
    """
    device = Xf.device
    N_img = Xf.size(0)
    N_txt = Tf.size(0)

    if caption_groups is None:
        assert N_txt % N_img == 0, "Cannot infer k; pass caption_groups or ensure N_txt = k * N_img"
        k = N_txt // N_img
        caption_groups = [list(range(i*k, (i+1)*k)) for i in range(N_img)]
    else:
        k = len(caption_groups[0])
        assert len(caption_groups) == N_img
        assert all(len(g) == k for g in caption_groups)
        flat = sorted([idx for g in caption_groups for idx in g])
        assert flat == list(range(N_txt)), "caption_groups must partition 0..N_txt-1"

    # base sims
    base_ii = Xf @ Xf.T           # (N_img, N_img) — Xf is already L2-normalized
    base_tt = Tf @ Tf.T           # (N_txt, N_txt)

    # Sii: replicate over each caption group j
    Sii = torch.empty((N_img, N_txt), device=device, dtype=base_ii.dtype)
    for j, group in enumerate(caption_groups):
        Sii[:, group] = base_ii[:, j:j+1]

    # Stt: average k×k for each (i,j), then broadcast across group j
    Stt = torch.empty((N_img, N_txt), device=device, dtype=base_tt.dtype)
    for i, gi in enumerate(caption_groups):
        mean_over_i = base_tt[gi, :].mean(dim=0)  # (N_txt,)
        for j, gj in enumerate(caption_groups):
            val = mean_over_i[gj].mean()
            Stt[i, gj] = val

    return Sii, Stt

@torch.no_grad()
def build_M(Sii: torch.Tensor, Stt: torch.Tensor, thr: ThresholdsST) -> torch.Tensor:
    """
    Build M using Sii and Stt:
        M = (Sii > p2) ∨ (Stt > p3)
    Output in {-1, +1}.
    """
    pos = (Sii > thr.p2) | (Stt > thr.p3)
    return pos.to(Sii.dtype).mul_(2.0).sub_(1.0)

class MultiPositiveBCELoss(torch.nn.Module):
    """
    ℓ = mean softplus( m_ij * ( -s_ij/τ + β ) ), where m_ij∈{-1,+1}, s_ij = Sit
    """
    def __init__(self, beta_init: float = -2.0, learn_beta: bool = True):
        super().__init__()
        if learn_beta:
            self.beta = torch.nn.Parameter(torch.tensor(float(beta_init)))
        else:
            self.register_buffer("beta", torch.tensor(float(beta_init)))

    def forward(self, Sit: torch.Tensor, M: torch.Tensor, tau: torch.Tensor) -> torch.Tensor:
        logits = M * (-Sit / tau + self.beta)
        return F.softplus(logits).mean()

    @torch.no_grad()
    def set_beta(self, value: float):
        if isinstance(self.beta, torch.nn.Parameter):
            self.beta.data.fill_(float(value))
        else:
            self.beta.copy_(torch.tensor(float(value), device=self.beta.device))

class MultiPositiveSoftBCELoss(nn.Module):
    """
add this
    """
    def __init__(self, beta_init: float = -2.0, learn_beta: bool = True):
        super().__init__()
        if learn_beta:
            self.beta = nn.Parameter(torch.tensor(float(beta_init)))
        else:
            self.register_buffer("beta", torch.tensor(float(beta_init)))

    def forward(self, Sit: torch.Tensor, W: torch.Tensor, tau: torch.Tensor):
        """
        Sit: (N_img, N_txt) cosine similarity
        W:   (N_img, N_txt) soft positives in [0,1]
        tau: scalar tensor
        """
        # logits = s/τ - β   (matches your existing sign convention)
        logits = Sit / tau - self.beta

        # Soft-target BCE: y*softplus(-logit) + (1-y)*softplus(logit)
        loss = W * F.softplus(-logits) + (1.0 - W) * F.softplus(logits)
        return loss.mean()


@torch.no_grad()
def estimate_beta_grid(
    batches,                 # iterable of tuples: (Xf, Tf, thr_st, align_kwargs)
    tau_scalar: float,
    beta_grid = tuple([-5.0 + 0.25*i for i in range(21)]),  # [-5, 0]
) -> float:
    """
    Grid-search β for the Sii/Stt-only mask:
      - Sit is used only inside the loss (s_ij)
      - M is built from Sii and Stt only
    Each item in `batches` is:
      (Xf, Tf, thr_st: ThresholdsST, align_kwargs: {"k": k} or {"caption_groups": ...})
    """
    best_beta, best_loss = None, float("inf")
    tau = torch.tensor(float(tau_scalar))

    for beta in beta_grid:
        tot, cnt = 0.0, 0
        for (Xf, Tf, thr_st, align_kwargs) in batches:
            # s_ij for the loss
            Sit = Xf @ Tf.t()  # (N_img, N_txt) — Xf, Tf assumed L2-normalized already

            # build M from Sii/Stt only
            Sii, Stt = _align_sims_k(Xf, Tf, **align_kwargs)
            M = build_M(Sii, Stt, thr_st)

            logits = M * (-Sit / tau + beta)
            loss = F.softplus(logits).mean().item()
            tot += loss; cnt += 1
        avg = tot / max(cnt, 1)
        if avg < best_loss:
            best_loss, best_beta = avg, beta
    return float(best_beta)

def contrastive_info_nce_loss(image_embeds: torch.Tensor,
                              text_embeds:  torch.Tensor,
                              labels:       torch.Tensor,
                              temperature:  float = 0.07,
                              classification_logits_image: torch.Tensor = None,
                              classification_logits_text:  torch.Tensor = None,
                              multi_label:  bool  = False,
                              lambda_class: float = 0) -> torch.Tensor:
    """
    Combined InfoNCE‐style contrastive loss + optional classification loss.
    
    Args:
        image_embeds: (B, D) normalized image embeddings
        text_embeds:  (B, D) normalized text embeddings
        labels:       (B, B) binary label matrix (if multi_label) 
                      or class indices (if single label).
        temperature:  scaling factor for contrastive logits.
        classification_logits_image: (B, C) logits for image→class (optional)
        classification_logits_text:  (B, C) logits for text→class (optional)
        multi_label:  whether classification is multi‐label (True) or single label (False)
        lambda_class: weight for classification loss component.
    
    Returns:
        loss: scalar tensor
    """
    # 1) Contrastive part (InfoNCE)
    # Compute logits = similarity(image, text) / temperature
    logits_it = torch.matmul(image_embeds, text_embeds.t()) / temperature  # (B, B)
    logits_ti = torch.matmul(text_embeds, image_embeds.t()) / temperature  # (B, B)
    
    # Define target for contrastive: for single‐label / one‐to‐one scenario:
    #   ground_truth = torch.arange(B, device=device)
    # But for multi‐label, maybe labels is binary matrix.
    B = image_embeds.size(0)
    device = image_embeds.device
    
    if not multi_label:
        target = torch.arange(B, dtype=torch.long, device=device)
        contrastive_loss_i2t = F.cross_entropy(logits_it, target)
        contrastive_loss_t2i = F.cross_entropy(logits_ti, target)
        contrastive_loss = (contrastive_loss_i2t + contrastive_loss_t2i) / 2.0
    else:
        # Multi‐label: flatten and use BCE with logits
        # For efficiency, treat logits_it and labels as matching pairs matrix
        loss_i2t = F.binary_cross_entropy_with_logits(logits_it, labels)
        loss_t2i = F.binary_cross_entropy_with_logits(logits_ti, labels)
        contrastive_loss = (loss_i2t + loss_t2i) / 2.0
        
    # Combine
    total_loss = contrastive_loss
    return total_loss

def evaluate(model, dataset, val_loader, config, device, soft_mask, loss_soft):
    model.eval()
    total_loss = 0.0
    n_batches = 0

    use_amp = (device == "cuda")

    for step, batch in enumerate(val_loader):
        with torch.no_grad():
            with torch.amp.autocast(device_type="cuda", enabled=use_amp, dtype=torch.bfloat16):
                input_images, input_texts, image_features = batch

                input_images = input_images.to(device)
                image_features = image_features.to(device)

                input_texts_tok = dataset.tokenizer.batch_encode_plus(
                    input_texts,
                    return_tensors="pt",
                    padding="max_length",
                    max_length=config.context_length,
                    truncation=True
                ).to(device)

                # same features as training
                text_features = model.get_text_embeddings(input_texts_tok).last_hidden_state[:, 0, :]
                image_embeds, text_embeds = model(input_images, input_texts_tok)

                # normalize projected embeddings
                image_embeds = F.normalize(image_embeds, dim=1)
                text_embeds  = F.normalize(text_embeds, dim=1)

                # normalize raw image/text features used for Sii/Stt only
                image_features_norm = F.normalize(image_features, dim=1)
                text_features_norm  = F.normalize(text_features, dim=1)

                # thresholds / tau
                if config.n_gpu == 1:
                    thres_image = model.thres_image
                    thres_text  = model.thres_text
                    tau = torch.exp(-model.logit_scale)
                else:
                    thres_image = model.module.thres_image
                    thres_text  = model.module.thres_text
                    tau = torch.exp(-model.module.logit_scale)

                # same alignment logic as training
                N_img = image_embeds.size(0)
                N_txt = text_embeds.size(0)

                align_kwargs = {}
                if N_txt % N_img == 0:
                    align_kwargs["k"] = N_txt // N_img

                if hasattr(dataset, "caption_groups_in_batch") and dataset.caption_groups_in_batch is not None:
                    align_kwargs = {"caption_groups": dataset.caption_groups_in_batch(step)}

                # Sit from projected embeddings, Sii/Stt from intra-modality raw features
                Sit = image_embeds @ text_embeds.t()
                Sii, Stt = _align_sims_k(image_features_norm, text_features_norm, **align_kwargs)

                W = soft_mask(Sii, Stt, p2=thres_image, p3=thres_text)
                loss = loss_soft(Sit, W, tau=tau)

                if config.n_gpu > 1:
                    loss = loss.mean()

                total_loss += float(loss.item())
                n_batches += 1

    model.train()
    return total_loss / max(1, n_batches)


set_seed(42)

In [ ]:
RUN_DIR = "../group_attn_run2" # change this to your own directory
import sys
import logging
from typing import Optional

def setup_logger(
    name: str,
    save_dir: Optional[str] = None,
    distributed_rank: int = 0,
    filename: str = "log.txt",
    level: int = logging.INFO,
) -> logging.Logger:
    """
    Create a logger that logs to stdout and optionally to a file.

    Args:
        name: Logger name (e.g. "MICCAI_Practice").
        save_dir: Directory for log file. If None, no file is written.
        distributed_rank: Only rank 0 writes to file and prints INFO+; other ranks can be WARN+.
        filename: Log filename inside save_dir.
        level: Base log level.

    Returns:
        A configured logging.Logger.
    """
    logger = logging.getLogger(name)

    # Prevent adding handlers multiple times (common in notebooks / repeated runs)
    if getattr(logger, "_is_configured", False):
        return logger

    logger.setLevel(level)
    logger.propagate = False  # don't double-log to root logger

    # Format: time | level | name | message
    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    # --- Stream handler (stdout) ---
    stream_handler = logging.StreamHandler(stream=sys.stdout)
    stream_handler.setFormatter(formatter)

    # Non-zero ranks: reduce noise
    if distributed_rank != 0:
        stream_handler.setLevel(logging.WARNING)
    else:
        stream_handler.setLevel(level)

    logger.addHandler(stream_handler)

    # --- File handler (rank 0 only) ---
    if save_dir is not None and distributed_rank == 0:
        os.makedirs(save_dir, exist_ok=True)
        log_path = os.path.join(save_dir, filename)

        file_handler = logging.FileHandler(log_path, mode="a")
        file_handler.setFormatter(formatter)
        file_handler.setLevel(level)
        logger.addHandler(file_handler)

    logger._is_configured = True
    return logger

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SoftGroupMask(nn.Module):
    """
    Build a differentiable "group" mask W in [0,1] from Sii and Stt.
    The basic idea of this is to turn the disease weighting into a continuous spectrum.
    
    Instead of:
        pos = (Sii > p2) | (Stt > p3)   -> hard OR
    we do:
        aF = sigmoid((Sii - p2)/gF)
        aT = sigmoid((Stt - p3)/gT)
        W  = soft_OR(aF, aT) = 1 - (1-aF)(1-aT)

    gF/gT control sharpness (smaller => more threshold-like).
    """
    def __init__(self, gF_init=0.02, gT_init=0.02, learn_g=True):
        super().__init__()
        gF = torch.tensor(float(gF_init))
        gT = torch.tensor(float(gT_init))
        if learn_g:
            self.log_gF = nn.Parameter(gF.log())
            self.log_gT = nn.Parameter(gT.log())
        else:
            self.register_buffer("log_gF", gF.log())
            self.register_buffer("log_gT", gT.log())

    def forward(self, Sii, Stt, p2, p3):
        gF = self.log_gF.exp().clamp_min(1e-6)
        gT = self.log_gT.exp().clamp_min(1e-6)

        aF = torch.sigmoid((Sii - p2) / gF)
        aT = torch.sigmoid((Stt - p3) / gT)

        # differentiable OR
        W  = 1.0 - (1.0 - aF) * (1.0 - aT)

        return W


## In the study, the vision encoder was trained, while text encoders remained frozen.

In [ ]:
unfreeze_vision = True
unfreeze_text = False
beta = -0.6319305409095584

In [ ]:
from omegaconf import OmegaConf
import tempfile

config = OmegaConf.create({
    "saved_checkpoints" : RUN_DIR,
    "image_feature_file" : './data/UKB/Macular_Measurement.csv',
    "train_json" : "captions_train.json",
    "val_json": "captions_val.json",
    "test_json" : "captions_test2.json",
    "logs": os.path.join(RUN_DIR, "logs"),
    "logs_name": "training_logs_assign.txt",
    "use_distance": False,
    "use_assign": True,
    "vision_model_checkpoint":'./RETFound_cfp_weights.pth', 
    "vision_model_output_dim": 1024,
    "text_model_output_dim": 1024, # change this into 1024
    "context_length": 512, 
    "proj_dim": 1024,
    "dropout": 0,
    "num_train_epochs": 2, 
    "batch_size": 16,
    "gradient_accumulation_steps" : 8,
    "loss": 'mutual',
    "optimizer": {
        "params":{
            'eps': 8.605285922082583e-07, 
            'lr': 0.00024276918679026895, 
            'weight_decay': 0.023262062576234904}
    },
    "n_gpu":1,
    "logging_steps" : 5,
    "save_steps" : 100,
    "val_steps":100
                          })

os.makedirs(config.saved_checkpoints, exist_ok=True)
os.makedirs(config.logs, exist_ok=True)

## Prepare the datasplit

In [ ]:
train_dataset = Fundus_RiskFactor_Dataset(config, 'train')
val_dataset = Fundus_RiskFactor_Dataset(config, 'val')

# Create the subset
#sampler = RandomSampler(dataset)
batch_size = config.batch_size
print('The total number of participants in the training set:', len(train_dataset))
print('The total number of participants in the validation set:', len(val_dataset))

In [ ]:
# Create DataLoaders
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
val_dataloader = torch.utils.data.DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

## Training Script

In [ ]:
# -------------------------------------------------
# Checkpoint Saving Function
# -------------------------------------------------
def save_checkpoint(config, epoch, global_step, model, optimizer):
    """
    Saves a checkpoint of the model and optimizer.

    Args:
        config (object): Configuration object containing paths and settings.
        epoch (int): Current training epoch.
        global_step (int): Current training step.
        model (torch.nn.Module): The model to save.
        optimizer (torch.optim.Optimizer): The optimizer to save.

    Notes:
        - Saves model and optimizer `state_dict()` along with epoch and training steps.
        - Handles multi-GPU setups by saving `model.module.state_dict()` when necessary.
        - Retries saving up to 10 times to handle potential I/O issues.
    """
    now = datetime.now()
    current_time_str = now.strftime("%H-%M-%S")  # avoid ":" in filenames
    checkpoint_path = os.path.join(
        config.saved_checkpoints,
        f'checkpoint_{epoch}_{global_step}_{current_time_str}.pt'
    )
    save_num = 0

    while save_num < 10:
        try:
            state_dict = {
                'epoch': epoch,
                'global_step': global_step,
                'model_state_dict': model.module.state_dict() if config.n_gpu > 1 else model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict()
            }
            torch.save(state_dict, checkpoint_path)
            logger.info(f"Checkpoint saved to {checkpoint_path}")
            break
        except Exception as e:
            logger.warning(f"Checkpoint save failed (attempt {save_num+1}/10): {e}")
            save_num += 1

    if save_num == 10:
        logger.info("Failed to save checkpoint after 10 attempts.")


# -------------------------------------------------
# Model, Optimizer, Scheduler, Loss Setup
# -------------------------------------------------
global logger

best_loss = 99999999.0      # legacy: not used now; kept if you reference it elsewhere
best_val_loss = float("inf")  # used for best checkpoint

model = Model_Wrapper(config)
model.freeze_model()

if unfreeze_vision:
    for child in model.vision_model.children():
        for param in child.parameters():
            param.requires_grad = True

if unfreeze_text:
    for child in model.text_model.children():
        for param in child.parameters():
            param.requires_grad = True

# Learnable thresholds (you already had these)
model.thres_image = nn.Parameter(torch.Tensor([0.9480642696527377]), requires_grad=False)
model.thres_text  = nn.Parameter(torch.Tensor([0.9808160442103382]), requires_grad=False)

# total training iterations
t_total = len(train_dataloader) // config.gradient_accumulation_steps * config.num_train_epochs

optimizer = AdamW(
    model.parameters(),
    lr=config.optimizer.params.lr,
    eps=config.optimizer.params.eps,
    weight_decay=config.optimizer.params.weight_decay
)

patience = 10  # not yet used for early stopping; you can wire this later if you want

# Warmup iterations = 20% of total iterations
num_warmup_steps = int(0.2 * t_total)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=t_total
)

device = "cuda" if torch.cuda.is_available() else "cpu"



# Create directories for saving checkpoints and logs
mkdir(path=config.saved_checkpoints)
mkdir(path=config.logs)

#Loss Objects
soft_mask = SoftGroupMask(gF_init=0.02, gT_init=0.02, learn_g=True).to(device)
loss_soft = MultiPositiveSoftBCELoss(beta_init=-2.0, learn_beta=True).to(device)


# Set up the logger for tracking training progress
logger = setup_logger("MICCAI_Practice", config.logs, 0, filename=config.logs_name)

global_step, global_loss, global_acc = 0, 0.0, 0.0
model = model.to(device)
model.zero_grad()


# -------------------------------------------------
# Logging of run configuration
# -------------------------------------------------
logger.info("***** Running training *****")
logger.info("\n ## Data & System Details ##")
logger.info("  Num examples = %d", len(train_dataset))
logger.info("  Num Epochs = %d", config.num_train_epochs)
logger.info("  Number of GPUs = %d", config.n_gpu)
logger.info("  Gradient Accumulation steps = %d", config.gradient_accumulation_steps)

logger.info("\n ## Optimization Details ##")
logger.info(
    "  Total train batch size (w. parallel, & accumulation) = %d",
    config.batch_size * config.gradient_accumulation_steps
)
logger.info("  Total optimization steps = %d", t_total)
logger.info("  Loss function = %s", config.loss)
logger.info("  Learning Rate = %f", config.optimizer.params.lr)
logger.info("  Weight Decay = %f", config.optimizer.params.weight_decay)

logger.info("\n ## Model Details ##")
logger.info("  Context length = %d", config.context_length)
logger.info("  Projection dimension = %d", config.proj_dim)

if scheduler:
    logger.info("  warmup steps = %d", num_warmup_steps)


# -------------------------------------------------
# Training Loop with Inline Validation
# -------------------------------------------------
for epoch in range(int(config.num_train_epochs)):
    model.train()
    logger.info(f"===== Epoch {epoch + 1}/{config.num_train_epochs} =====")

    for step, batch in enumerate(train_dataloader):
        input_images, input_texts, image_features = batch

        # Move inputs to device
        input_images = input_images.to(device)
        image_features = image_features.to(device)
        input_texts_tok = train_dataset.tokenizer.batch_encode_plus(
            input_texts,
            return_tensors='pt',
            padding='max_length',
            max_length=config.context_length,
            truncation=True
        ).to(device)

        # Extract text features (CLS)
        text_features = model.get_text_embeddings(input_texts_tok).last_hidden_state[:, 0, :].to(device)

        # Get projected embeddings from the model
        image_embeds, text_embeds = model(input_images, input_texts_tok)

        # Normalize embeddings for similarity calculations
        image_embeds = image_embeds / image_embeds.norm(dim=1, keepdim=True)
        text_embeds  = text_embeds  / text_embeds.norm(dim=1, keepdim=True)

        # Normalize raw image and text features
        text_features_norm  = text_features  / text_features.norm(dim=1, keepdim=True)
        image_features_norm = image_features / image_features.norm(dim=1, keepdim=True)

        # Define logit scale and thresholds for image-text similarity
        if config.n_gpu == 1:
            logit_scale = model.logit_scale.exp()
            thres_image = model.thres_image
            thres_text  = model.thres_text
            beta        = model.beta
        else:
            logit_scale = model.module.logit_scale.exp()
            thres_image = model.module.thres_image
            thres_text  = model.module.thres_text
            beta        = model.module.beta

        # Compute similarity logits (not directly used in loss here, but kept)
        logits_per_image = logit_scale * image_embeds @ text_embeds.t()
        logits_per_text  = logit_scale * text_embeds  @ image_embeds.t()

        # ------------------------
        # Compute loss (mutual)
        # ------------------------
        if config.loss == 'mutual':
            Sit = image_embeds @ text_embeds.t()

            # CLIP-style temperature tau
            tau = torch.exp(-model.logit_scale) if config.n_gpu == 1 else torch.exp(-model.module.logit_scale)

            N_img = image_embeds.size(0)
            N_txt = text_embeds.size(0)

            # Alignment kwargs
            align_kwargs = {}
            if N_txt % N_img == 0:
                align_kwargs["k"] = N_txt // N_img

            if hasattr(train_dataset, "caption_groups_in_batch") and train_dataset.caption_groups_in_batch is not None:
                align_kwargs = {"caption_groups": train_dataset.caption_groups_in_batch(step)}

            # Build Sii/Stt
            Sii, Stt = _align_sims_k(image_features_norm, text_features_norm, **align_kwargs)

            # Thresholds for M
            thr = Thresholds(
                p2=model.thres_image,   # Sii >
                p3=model.thres_text     # Stt >
            )

            # 1) Build soft attention weights W in [0,1]
            W = soft_mask(Sii, Stt, p2=model.thres_image, p3=model.thres_text)
            
            # 2) Use soft-target loss
            loss = loss_soft(Sit, W, tau=tau)

        else:
            raise ValueError(f"Unsupported loss type: {config.loss}")

        # Handle multi-GPU training and gradient accumulation
        if config.n_gpu > 1:
            loss = loss.mean()
        if config.gradient_accumulation_steps > 1:
            loss = loss / config.gradient_accumulation_steps

        # Backpropagation
        loss.backward()
        global_loss += loss.item()

        # Perform optimization step
        if (step + 1) % config.gradient_accumulation_steps == 0:
            global_step += 1
            optimizer.step()  # Update model parameters

            # Ensure logit scaling does not exceed 100 (as per CLIP paper)
            if config.n_gpu == 1:
                model.logit_scale.data = torch.clamp(model.logit_scale.data, 0, 4.6052)
            else:
                model.module.logit_scale.data = torch.clamp(model.module.logit_scale.data, 0, 4.6052)

            # Update learning rate scheduler
            if scheduler:
                scheduler.step()

            model.zero_grad()  # Reset gradients

            # Logging training progress
            if global_step % config.logging_steps == 0:
                logger.info(
                    "Epoch: {}, global_step: {}, lr: {:.7f}, loss: {:.7f} ({:.7f})".format(
                        epoch, global_step,
                        optimizer.param_groups[0]["lr"],
                        loss.item(), global_loss / global_step
                    )
                )
            # (Optional) step-based checkpointing — if you still want it
            if (config.save_steps > 0 and global_step % config.save_steps == 0) or global_step == t_total:
                save_checkpoint(config, epoch, global_step, model, optimizer)

            # -------------------------------------------------
            # INLINE VALIDATION LOOP (every config.val_steps)
            # -------------------------------------------------
            if global_step % config.val_steps == 0:
                model.eval()
                val_loss = 0.0
                val_steps = 0

                avg_val_loss = evaluate(
                    model=model,
                    dataset=val_dataset,
                    val_loader=val_dataloader,
                    config=config,
                    device=device,
                    soft_mask=soft_mask,
                    loss_soft=loss_soft,
                )

                logger.info(f"[Validation @ global_step {global_step}] avg_val_loss = {avg_val_loss:.6f}")
                # Save best model
                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    logger.info(
                        f"Validation loss improved to {best_val_loss:.6f}. Saving best checkpoint."
                    )
                    save_checkpoint(config, epoch, global_step, model, optimizer)

                model.train()